# Clean RCMFP enzyme-retrieval workflow

This notebook implements the finalized workflow from the DORAnet/DORA-XGB → atom mapping → RCMFP → known enzyme retrieval discussion. It uses `ruleBase` as the practical bridge between expanded DORAnet rule names such as `rule0003_170` and known-reference coarse operator IDs such as `rule0003`, then ranks candidate natural enzyme precedents by RCMFP Tanimoto similarity.

Required objects before running: `reactionWithRuleDF`, `dataDir`, and `DNADesignResultsDir`.

In [ ]:
import os, sys, ast
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem, rdBase
from rdkit.Chem import rdmolfiles

# User paths
ERGOCHEMICS_SRC = "/users/sghosh6/DTRA_project/MACAW/ergochemics/src"
if ERGOCHEMICS_SRC not in sys.path: sys.path.insert(0, ERGOCHEMICS_SRC)

import ergochemics.mapping as egmap
import ergochemics.similarity as egsim

outputDir = Path(DNADesignResultsDir); outputDir.mkdir(parents=True, exist_ok=True)
knownEnzymeParquetPath = Path(dataDir) / "DORAnet" / "known_enzyme_reactions_union.parquet"
knownRcmfpCachePath = Path(dataDir) / "DORAnet" / "known_rxn_rcmfp_cache.npz"
if not knownEnzymeParquetPath.exists(): knownEnzymeParquetPath = Path("/mnt/data/known_enzyme_reactions_union.parquet")
if not knownRcmfpCachePath.exists(): knownRcmfpCachePath = Path("/mnt/data/known_rxn_rcmfp_cache.npz")

TOP_K, MIN_SIMILARITY, FP_SIDE_LENGTH, FP_RADIUS = 20, 0.0, 2048, 2
print("RDKit version:", rdBase.rdkitVersion); print("ergochemics mapping file:", egmap.__file__)

## 1. Patch RDKit/ergochemics compatibility

This preserves atom-map labels. Do not use a patch that clears atom-map numbers, because RCMFP needs atom maps to infer reaction centers.

In [ ]:
_RDKit_MolToSmiles_original = rdmolfiles.MolToSmiles

def MolToSmiles_compat(mol, *args, **kwargs):
    kwargs.pop("ignoreAtomMapNumbers", None)
    return _RDKit_MolToSmiles_original(mol, *args, **kwargs)

Chem.MolToSmiles = MolToSmiles_compat; egmap.Chem.MolToSmiles = MolToSmiles_compat; egsim.Chem.MolToSmiles = MolToSmiles_compat
operator_map_reaction, get_reaction_center = egmap.operator_map_reaction, egmap.get_reaction_center
ReactionFingerprinter, MolFeaturizer = egsim.ReactionFingerprinter, egsim.MolFeaturizer
print("Patch test:", Chem.MolToSmiles(Chem.MolFromSmiles("[CH3:1][OH:2]"), ignoreAtomMapNumbers=True))

## 2. Normalize DORAnet reaction strings and atom-map query reactions

In [ ]:
def normalize_reaction_string(rxn):
    if pd.isna(rxn): return None
    rxn = str(rxn).strip().replace(" ", "")
    if rxn.count(">") == 2 and ">>" in rxn: return rxn
    parts = rxn.split(">")
    return f"{parts[0]}>>{parts[2]}" if len(parts) == 3 else None

def map_query_reaction_with_rule(row):
    rxn, ruleSMARTS = row.get("queryReaction"), row.get("ruleSMARTS")
    if pd.isna(rxn) or str(rxn).strip() == "": return None, "missing_queryReaction"
    if pd.isna(ruleSMARTS) or str(ruleSMARTS).strip() == "": return None, "missing_ruleSMARTS"
    rxn, ruleSMARTS, lastError = str(rxn).replace(" ", ""), str(ruleSMARTS).strip(), None
    if ">>" not in rxn: return None, "invalid_queryReaction_no_double_arrow"
    if ">>" not in ruleSMARTS: return None, "invalid_ruleSMARTS_no_double_arrow"
    for explicitHsFlag, statusLabel in [(False, "mapped"), (True, "mapped_explicit_h")]:
        try:
            result = operator_map_reaction(rxn=rxn, operator=ruleSMARTS, explicit_hs=explicitHsFlag, quiet=True)
            if result.did_map and result.atom_mapped_smarts is not None: return result.atom_mapped_smarts, statusLabel
            lastError = f"{statusLabel}_failed"
        except Exception as exc: lastError = f"mapping_error:{type(exc).__name__}:{exc}"
    return None, lastError

queryDF = reactionWithRuleDF.copy()
queryDF["queryReaction"] = queryDF["reactionString"].apply(normalize_reaction_string)
queryDF = queryDF[queryDF["queryReaction"].notna() & queryDF["ruleSMARTS"].notna()].drop(columns=["queryMappedReaction", "rcmfpMappingStatus"], errors="ignore").copy()

mappedPairs = queryDF.progress_apply(map_query_reaction_with_rule, axis=1)
queryDF[["queryMappedReaction", "rcmfpMappingStatus"]] = pd.DataFrame(mappedPairs.tolist(), index=queryDF.index)
mappedQueryDF = queryDF[queryDF["queryMappedReaction"].notna()].copy(); failedMappingDF = queryDF[queryDF["queryMappedReaction"].isna()].copy()
print(queryDF["rcmfpMappingStatus"].value_counts(dropna=False)); print("Successful atom mapping:", len(mappedQueryDF), "| Failed:", len(failedMappingDF))
failedMappingDF.to_csv(outputDir / "RCMFP_query_mapping_failures.csv", index=False)

## 3. Compute query-side RCMFP fingerprints

In [ ]:
reactionFingerprinter = ReactionFingerprinter(radius=FP_RADIUS, length=FP_SIDE_LENGTH, mol_featurizer=MolFeaturizer())

def compute_rcmfp_with_status(mappedRxn):
    try:
        if pd.isna(mappedRxn) or str(mappedRxn).strip() == "": return None, "missing_mapped_reaction"
        mappedRxn = str(mappedRxn).replace(" ", "")
        if ">>" not in mappedRxn: return None, "invalid_mapped_reaction_no_double_arrow"
        if not pd.Series([mappedRxn]).str.contains(r":\d+\]", regex=True).iloc[0]: return None, "no_atom_map_labels"
        lrc, rrc = get_reaction_center(mappedRxn, mode="combined")
        if len(lrc) == 0 or len(rrc) == 0: return None, f"empty_reaction_center:lrc={len(lrc)}:rrc={len(rrc)}"
        fp = reactionFingerprinter.fingerprint(mappedRxn, output_type="bit", use_rc=True, rc_dist_ub=None)
        return fp.astype(bool), "rcmfp_success"
    except Exception as exc: return None, f"rcmfp_error:{type(exc).__name__}:{exc}"

rcmfpPairs = mappedQueryDF["queryMappedReaction"].progress_apply(compute_rcmfp_with_status)
mappedQueryDF[["queryRCMFP", "rcmfpStatus"]] = pd.DataFrame(rcmfpPairs.tolist(), index=mappedQueryDF.index)
queryRcmfpFailureDF = mappedQueryDF[mappedQueryDF["queryRCMFP"].isna()].copy()
mappedQueryDF = mappedQueryDF[mappedQueryDF["queryRCMFP"].notna()].copy().reset_index(drop=True)
print(mappedQueryDF["rcmfpStatus"].value_counts(dropna=False)); print("Query reactions with valid RCMFP:", len(mappedQueryDF))
queryRcmfpFailureDF.to_csv(outputDir / "RCMFP_query_fingerprint_failures.csv", index=False)

## 4. Load and align known enzyme reference metadata and RCMFP cache

In [ ]:
knownEnzymeAllDF = pd.read_parquet(knownEnzymeParquetPath)
knownRcmfpCache = np.load(knownRcmfpCachePath, allow_pickle=True)
knownFpMatrix = knownRcmfpCache["fingerprints"].astype(bool)
knownRcPatternLHS, knownRcPatternRHS, knownValidIndices = knownRcmfpCache["rc_patterns_lhs"], knownRcmfpCache["rc_patterns_rhs"], knownRcmfpCache["valid_indices"]
knownEnzymeReferenceDF = knownEnzymeAllDF.iloc[knownValidIndices].reset_index(drop=True).copy()
knownEnzymeReferenceDF["knownRcPatternLHS"], knownEnzymeReferenceDF["knownRcPatternRHS"], knownEnzymeReferenceDF["knownOriginalIndex"] = knownRcPatternLHS, knownRcPatternRHS, knownValidIndices
assert len(knownEnzymeReferenceDF) == knownFpMatrix.shape[0]
print("Known enzyme reference:", knownEnzymeReferenceDF.shape); print("Known FP matrix:", knownFpMatrix.shape)

## 5. Prepare query metadata and fingerprint matrices

In [ ]:
def make_reverse_fp_matrix(fpMatrix, sideLength=FP_SIDE_LENGTH): return np.hstack([fpMatrix[:, sideLength:], fpMatrix[:, :sideLength]]).astype(bool)
def make_fp_matrix_from_column(df, fpCol):
    validMask = df[fpCol].notna(); metaDF = df.loc[validMask].drop(columns=[fpCol], errors="ignore").reset_index(drop=True)
    fpMatrix = np.vstack(df.loc[validMask, fpCol].to_numpy()).astype(bool); fpReverseMatrix = make_reverse_fp_matrix(fpMatrix)
    return metaDF, fpMatrix, fpReverseMatrix, fpMatrix.sum(axis=1), fpReverseMatrix.sum(axis=1)

queryMetaDF, queryFpMatrix, queryFpMatrixReverse, queryPopcounts, queryReversePopcounts = make_fp_matrix_from_column(mappedQueryDF, "queryRCMFP")
knownFpMatrixReverse = make_reverse_fp_matrix(knownFpMatrix)
knownPopcounts, knownReversePopcounts = knownFpMatrix.sum(axis=1), knownFpMatrixReverse.sum(axis=1)

queryKeepCols = ["reactants", "products", "reactionString", "ruleName", "reactionType", "rxn_str", "ruleSMARTS", "candidateUniProtRaw", "numCandidateUniProt", "hasRuleLookup", "queryReaction", "queryMappedReaction", "rcmfpMappingStatus", "rcmfpStatus", "feasibilityScore_rule1", "feasibilityLabel_rule1", "feasibilityScore_rule2", "feasibilityLabel_rule2", "feasibilityScore_rule3", "feasibilityLabel_rule3", "feasibilityScore_rule4", "feasibilityLabel_rule4"]
queryMetaDF = queryMetaDF[[c for c in queryKeepCols if c in queryMetaDF.columns]].copy()
print("Query metadata:", queryMetaDF.shape); print("Query FP matrix:", queryFpMatrix.shape); print("Known FP matrix:", knownFpMatrix.shape)

## 6. Add `ruleBase` to known reference and query metadata

This avoids forcing known reference rules into expanded DORAnet rule names. It uses coarse operator-family IDs such as `rule0003` and then lets RCMFP decide the closest natural reaction within that family.

In [ ]:
def get_rule_base(x): return None if pd.isna(x) else str(x).strip().split("_")[0]
def parse_operator_list(x):
    if isinstance(x, np.ndarray): return [str(v).strip() for v in x.tolist() if v is not None and str(v).strip()]
    if isinstance(x, (list, tuple, set)): return [str(v).strip() for v in list(x) if v is not None and str(v).strip()]
    if x is None or pd.isna(x): return []
    s = str(x).strip()
    if s in ["", "nan", "None", "[]"]: return []
    try:
        parsed = ast.literal_eval(s)
        return [str(v).strip() for v in list(parsed)] if isinstance(parsed, (list, tuple, set, np.ndarray)) else [str(parsed).strip()]
    except Exception:
        for sep in ["|", ";", ","]:
            if sep in s: return [v.strip().strip("'\"") for v in s.split(sep) if v.strip()]
        return [s]

def choose_known_rule_base(row):
    top = row.get("top_mapped_operator")
    if top is not None and not pd.isna(top) and str(top).strip().startswith("rule"): return str(top).strip(), "top_mapped_operator"
    for op in parse_operator_list(row.get("all_mapped_operators")):
        if str(op).startswith("rule"): return str(op), "all_mapped_operators"
    return None, "no_ruleBase"

knownEnzymeAllDF = knownEnzymeAllDF.copy()
if "knownOriginalIndex" not in knownEnzymeAllDF.columns: knownEnzymeAllDF["knownOriginalIndex"] = knownEnzymeAllDF.index
ruleBasePairs = knownEnzymeAllDF.apply(choose_known_rule_base, axis=1)
knownEnzymeAllDF["ruleBase"], knownEnzymeAllDF["ruleBaseSource"] = [x[0] for x in ruleBasePairs], [x[1] for x in ruleBasePairs]

knownEnzymeReferenceDF = knownEnzymeReferenceDF.drop(columns=[c for c in ["ruleBase", "ruleBaseSource"] if c in knownEnzymeReferenceDF.columns], errors="ignore")
knownEnzymeReferenceDF = knownEnzymeReferenceDF.merge(knownEnzymeAllDF[["knownOriginalIndex", "ruleBase", "ruleBaseSource"]], on="knownOriginalIndex", how="left")
queryMetaDF = queryMetaDF.reset_index(drop=True).copy(); knownMetaDF = knownEnzymeReferenceDF.reset_index(drop=True).copy()
queryMetaDF["queryRowId"], knownMetaDF["knownRowId"] = np.arange(len(queryMetaDF)), np.arange(len(knownMetaDF))
queryMetaDF["ruleBase"], knownMetaDF["_ruleKey"], queryMetaDF["_ruleKey"] = queryMetaDF["ruleName"].apply(get_rule_base), knownMetaDF["ruleBase"].astype("object"), queryMetaDF["ruleBase"].astype("object")
knownIndicesByRuleBase = {ruleBase: idx.to_numpy() for ruleBase, idx in knownMetaDF.dropna(subset=["_ruleKey"]).groupby("_ruleKey").groups.items()}
print("Known ruleBase source:\n", knownMetaDF["ruleBaseSource"].value_counts(dropna=False)); print("Overlapping ruleBase:", len(set(queryMetaDF["ruleBase"].dropna()).intersection(set(knownMetaDF["ruleBase"].dropna()))))

## 7. Run ruleBase-constrained RCMFP enzyme retrieval

In [ ]:
def tanimoto_packed_one(queryPackedRow, queryPop, refPackedMatrix, refPopcounts, popcount8):
    inter = popcount8[np.bitwise_and(refPackedMatrix, queryPackedRow)].sum(axis=1).astype(np.float32); union = refPopcounts + queryPop - inter
    scores = np.zeros(len(refPopcounts), dtype=np.float32); valid = union > 0; scores[valid] = inter[valid] / union[valid]
    return scores

def retrieve_top_hits_for_query(queryIdx, topK=TOP_K, minSimilarity=MIN_SIMILARITY):
    queryRuleBase = queryMetaDF.loc[queryIdx, "_ruleKey"]
    candidateIdx, searchMode = (knownIndicesByRuleBase[queryRuleBase], "same_ruleBase") if pd.notna(queryRuleBase) and queryRuleBase in knownIndicesByRuleBase else (np.arange(len(knownMetaDF)), "full_database_fallback")
    qPacked, qPop = queryPacked[queryIdx], queryPopcounts[queryIdx]
    scores = np.maximum(tanimoto_packed_one(qPacked, qPop, knownPacked[candidateIdx], knownPopcounts[candidateIdx], popcount8), tanimoto_packed_one(qPacked, qPop, knownPackedReverse[candidateIdx], knownReversePopcounts[candidateIdx], popcount8))
    validLocal = np.where(scores > minSimilarity)[0]
    if len(validLocal) == 0: return []
    topLocal = validLocal[np.argpartition(-scores[validLocal], topK - 1)[:topK]] if len(validLocal) > topK else validLocal
    topLocal = topLocal[np.argsort(-scores[topLocal])]
    return [{"queryRowId": int(queryIdx), "rank": rank, "knownRowId": int(candidateIdx[localIdx]), "rcmfpSimilarity": float(scores[localIdx]), "searchMode": searchMode, "numCandidatesSearched": int(len(candidateIdx))} for rank, localIdx in enumerate(topLocal, start=1)]

queryFpMatrix, knownFpMatrix, knownFpMatrixReverse = queryFpMatrix.astype(bool), knownFpMatrix.astype(bool), knownFpMatrixReverse.astype(bool)
queryPacked, knownPacked, knownPackedReverse = np.packbits(queryFpMatrix.astype(np.uint8), axis=1), np.packbits(knownFpMatrix.astype(np.uint8), axis=1), np.packbits(knownFpMatrixReverse.astype(np.uint8), axis=1)
queryPopcounts, knownPopcounts, knownReversePopcounts = queryFpMatrix.sum(axis=1).astype(np.int32), knownFpMatrix.sum(axis=1).astype(np.int32), knownFpMatrixReverse.sum(axis=1).astype(np.int32)
popcount8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

allHitRecords = []
for queryIdx in tqdm(range(len(queryMetaDF)), desc="RCMFP enzyme retrieval"): allHitRecords.extend(retrieve_top_hits_for_query(queryIdx))
rcmfpHitsLongDF = pd.DataFrame(allHitRecords)
print("Total hits:", len(rcmfpHitsLongDF)); print(rcmfpHitsLongDF["searchMode"].value_counts(dropna=False))

## 8. Build full summary, filter Moderate/Strong, deduplicate, and save outputs

In [ ]:
def classify_precedent(sim):
    if pd.isna(sim): return "No natural precedent retrieved"
    if sim >= 0.60: return "Strong natural enzyme precedent"
    if sim >= 0.35: return "Moderate natural enzyme precedent"
    if sim > 0: return "Weak precedent; likely enzyme-engineering risk"
    return "No useful RCMFP precedent"

queryDisplayCols = [c for c in ["queryRowId", "reactionString", "ruleName", "ruleBase", "reactionType", "queryReaction", "queryMappedReaction", "candidateUniProtRaw", "numCandidateUniProt", "rcmfpMappingStatus", "rcmfpStatus", "feasibilityScore_rule1", "feasibilityLabel_rule1", "feasibilityScore_rule2", "feasibilityLabel_rule2", "feasibilityScore_rule3", "feasibilityLabel_rule3", "feasibilityScore_rule4", "feasibilityLabel_rule4"] if c in queryMetaDF.columns]
knownDisplayCols = [c for c in ["knownRowId", "knownOriginalIndex", "ruleBase", "ruleBaseSource", "rxn_idx", "mapped", "unmapped", "orig_rxn_text", "rule", "source", "quality", "natural", "organism", "protein_refs", "protein_db", "ec_num", "top_mapped_operator", "all_mapped_operators"] if c in knownMetaDF.columns]
rcmfpHitsLongDF = rcmfpHitsLongDF.merge(queryMetaDF[queryDisplayCols], on="queryRowId", how="left", suffixes=("", "_query")).merge(knownMetaDF[knownDisplayCols], on="knownRowId", how="left", suffixes=("", "_known")).sort_values(["queryRowId", "rank"]).reset_index(drop=True)

bestHitDF = rcmfpHitsLongDF.sort_values(["queryRowId", "rank"]).groupby("queryRowId", as_index=False).first()
bestHitCols = [c for c in ["queryRowId", "rcmfpSimilarity", "searchMode", "numCandidatesSearched", "knownRowId", "knownOriginalIndex", "ruleBase_known", "ruleBaseSource", "rxn_idx", "mapped", "unmapped", "orig_rxn_text", "rule", "source", "quality", "natural", "organism", "protein_refs", "protein_db", "ec_num", "top_mapped_operator", "all_mapped_operators"] if c in bestHitDF.columns]
queryRcmfpSummaryDF_full = queryMetaDF.merge(bestHitDF[bestHitCols], on="queryRowId", how="left").rename(columns={"rcmfpSimilarity": "rcmfpBestSimilarity", "searchMode": "rcmfpSearchMode", "numCandidatesSearched": "rcmfpNumCandidatesSearched", "knownRowId": "rcmfpTopKnownRowId", "knownOriginalIndex": "rcmfpTopKnownOriginalIndex", "ruleBase_known": "rcmfpTopKnownRuleBase", "ruleBaseSource": "rcmfpTopRuleBaseSource", "rxn_idx": "rcmfpTopRxnIdx", "mapped": "rcmfpTopMappedReaction", "unmapped": "rcmfpTopUnmappedReaction", "orig_rxn_text": "rcmfpTopOriginalReactionText", "rule": "rcmfpTopKnownRuleSMARTS", "source": "rcmfpTopSource", "quality": "rcmfpTopQuality", "natural": "rcmfpTopNatural", "organism": "rcmfpTopOrganism", "protein_refs": "rcmfpTopProteinRefs", "protein_db": "rcmfpTopProteinDB", "ec_num": "rcmfpTopEC", "top_mapped_operator": "rcmfpTopMappedOperator", "all_mapped_operators": "rcmfpTopAllMappedOperators"})
queryRcmfpSummaryDF_full["rcmfpNumHits"] = rcmfpHitsLongDF.groupby("queryRowId").size().reindex(queryRcmfpSummaryDF_full["queryRowId"]).fillna(0).astype(int).to_numpy()

feasibilityCols = [c for c in ["feasibilityScore_rule1", "feasibilityScore_rule2", "feasibilityScore_rule3", "feasibilityScore_rule4"] if c in queryRcmfpSummaryDF_full.columns]
queryRcmfpSummaryDF_full["bestDORAXGBFeasibility"] = queryRcmfpSummaryDF_full[feasibilityCols].apply(pd.to_numeric, errors="coerce").max(axis=1) if feasibilityCols else np.nan
queryRcmfpSummaryDF_full["rcmfpBestSimilarity"] = pd.to_numeric(queryRcmfpSummaryDF_full["rcmfpBestSimilarity"], errors="coerce")
queryRcmfpSummaryDF_full["enzymeImplementationScore"] = 0.60 * queryRcmfpSummaryDF_full["bestDORAXGBFeasibility"].fillna(0) + 0.40 * queryRcmfpSummaryDF_full["rcmfpBestSimilarity"].fillna(0)
queryRcmfpSummaryDF_full["enzymePrecedentCategory"] = queryRcmfpSummaryDF_full["rcmfpBestSimilarity"].apply(classify_precedent)
queryRcmfpSummaryDF_full = queryRcmfpSummaryDF_full.sort_values("enzymeImplementationScore", ascending=False).reset_index(drop=True)

keepCategories = ["Moderate natural enzyme precedent", "Strong natural enzyme precedent"]
queryRcmfpSummaryDF_moderateStrong = queryRcmfpSummaryDF_full[queryRcmfpSummaryDF_full["enzymePrecedentCategory"].isin(keepCategories)].copy().reset_index(drop=True)
queryRcmfpSummaryDF_moderateStrong_unique = queryRcmfpSummaryDF_moderateStrong.sort_values("enzymeImplementationScore", ascending=False).drop_duplicates(subset=["queryReaction", "ruleName"], keep="first").reset_index(drop=True)

knownEnzymeReferenceDF.to_parquet(outputDir / "knownEnzymeReferenceDF_with_ruleBase.parquet", index=False)
rcmfpHitsLongDF.to_csv(outputDir / "RCMFP_enzyme_retrieval_hits_long_ruleBase.csv", index=False)
queryRcmfpSummaryDF_full.to_csv(outputDir / "reactionWithRuleDF_RCMFP_summary_full_ruleBase.csv", index=False)
queryRcmfpSummaryDF_moderateStrong.to_csv(outputDir / "reactionWithRuleDF_RCMFP_summary_moderateStrong_ruleBase.csv", index=False)
queryRcmfpSummaryDF_moderateStrong_unique.to_csv(outputDir / "reactionWithRuleDF_RCMFP_summary_moderateStrong_unique_ruleBase.csv", index=False)
queryRcmfpSummaryDF_moderateStrong_unique.to_parquet(outputDir / "reactionWithRuleDF_RCMFP_summary_moderateStrong_unique_ruleBase.parquet", index=False)

print("Full summary rows:", len(queryRcmfpSummaryDF_full)); print("Moderate/Strong rows:", len(queryRcmfpSummaryDF_moderateStrong)); print("Unique Moderate/Strong rows:", len(queryRcmfpSummaryDF_moderateStrong_unique))
print("\nSearch mode counts:\n", queryRcmfpSummaryDF_full["rcmfpSearchMode"].value_counts(dropna=False))
print("\nPrecedent categories:\n", queryRcmfpSummaryDF_full["enzymePrecedentCategory"].value_counts(dropna=False))
print("\nRCMFP similarity summary:\n", queryRcmfpSummaryDF_full["rcmfpBestSimilarity"].describe())

## 9. Inspect final prioritized candidates

In [ ]:
displayCols = [c for c in ["reactionString", "ruleName", "ruleBase", "bestDORAXGBFeasibility", "rcmfpBestSimilarity", "rcmfpSearchMode", "rcmfpTopKnownRuleBase", "rcmfpTopEC", "rcmfpTopOrganism", "enzymePrecedentCategory", "enzymeImplementationScore"] if c in queryRcmfpSummaryDF_moderateStrong_unique.columns]
queryRcmfpSummaryDF_moderateStrong_unique[displayCols].head(20)